# Support Triage Agent — LangGraph Flow Demo

Builds the actual compiled `StateGraph` (`src/graph/build_graph.py`) and
streams **one ticket** through it live — a real `graph.stream(...)` call
against the Groq API, printing each node's output the instant it's
produced.

In [1]:
import sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Works whether Jupyter's cwd is notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config.settings import get_settings
from src.graph.build_graph import build_default_deps, build_graph
from src.main import load_tickets

settings = get_settings()
print(f"Repo root: {REPO_ROOT}")
print(f"Model: {settings.groq_model}")

/Users/mahendarprakash/SupportTriageAgent/.supportvenv/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Repo root: /Users/mahendarprakash/SupportTriageAgent
Model: llama-3.3-70b-versatile


## Building the graph

`auto_approve=True` mirrors how `python -m src.main --all --auto-approve`
runs the pipeline: `hitl_gate` auto-approves instead of blocking on an
`input()` prompt, which would otherwise hang a notebook cell forever.

In [2]:
deps = build_default_deps(auto_approve=True, interactive=False, settings=settings)
graph = build_graph(deps)

g = graph.get_graph()
real_nodes = [n for n in g.nodes if n not in ("__start__", "__end__")]
print(f"{len(real_nodes)} nodes: {real_nodes}\n")

print("Edges:")
seen = set()
for edge in g.edges:
    key = (edge.source, edge.target)
    if key in seen:
        continue
    seen.add(key)
    label = f"  {edge.source} -> {edge.target}"
    if edge.conditional:
        label += "   (conditional)"
    print(label)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: support-triage-agent
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



8 nodes: ['ingest', 'sentiment_policy_check', 'rag_retrieve', 'draft_answer', 'route', 'confidence_recheck', 'hitl_gate', 'audit_log']

Edges:
  __start__ -> ingest
  audit_log -> __end__
  draft_answer -> route
  hitl_gate -> audit_log
  ingest -> sentiment_policy_check
  rag_retrieve -> draft_answer
  sentiment_policy_check -> rag_retrieve
  route -> confidence_recheck   (conditional)
  route -> hitl_gate   (conditional)
  confidence_recheck -> rag_retrieve   (conditional)
  confidence_recheck -> hitl_gate   (conditional)


This matches the wiring in `build_graph`: a linear
`ingest -> sentiment_policy_check -> rag_retrieve -> draft_answer -> route`
spine, then a conditional branch — only `AUTO_RESOLVE` drafts go through
`confidence_recheck`'s stricter LLM-as-judge pass, which can loop back to
`rag_retrieve` with a reformulated query (up to
`max_retrieval_attempts`), while every other route goes straight to
`hitl_gate` -> `audit_log`.

## The ticket

`TCK-1016` — a clean, in-policy plan-switch request. It isn't one of the
four tickets `single_ticket_walkthrough.ipynb` covers, and its
`AUTO_RESOLVE` outcome means it's the one ticket type that actually
reaches every node in the graph, including the `confidence_recheck` loop.

In [3]:
tickets_by_id = {t.ticket_id: t for t in load_tickets(settings)}
ticket = tickets_by_id["TCK-1016"]
ticket.model_dump()

{'ticket_id': 'TCK-1016',
 'customer_id': 'CUST-016',
 'subject': 'Switch to monthly plan',
 'message': "I'd like to switch my subscription (SUB-5567) from annual to monthly billing.",
 'conversation_history': [],
 'priority': 'low',
 'category': 'subscription_cancellation',
 'order_id': None,
 'account_id': None,
 'subscription_id': 'SUB-5567',
 'days_since_purchase': None,
 'previous_refund_request_count': 0,
 'days_since_last_refund_request': None}

## Streaming the flow, node by node

`graph.stream(..., stream_mode="updates")` yields one `{node_name: delta}`
dict per node as soon as it finishes — the live equivalent of watching
`ARCHITECTURE.md`'s pipeline diagram execute in real time.

In [4]:
final_state = {"ticket": ticket}
t0 = time.perf_counter()

for step in graph.stream({"ticket": ticket}, stream_mode="updates"):
    for node_name, update in step.items():
        update = update or {}
        elapsed = time.perf_counter() - t0
        print(f"[{elapsed:5.2f}s] ---- {node_name} ----")
        for key, value in update.items():
            print(f"    {key}: {value!r}")
        final_state.update(update)

print(f"\nTotal: {time.perf_counter() - t0:.2f}s")

[ 0.00s] ---- ingest ----
    retrieval_attempts: 0
    conversation_context: '(no conversation history)'


[ 0.30s] ---- sentiment_policy_check ----
    abuse_detected: False
    sentiment: 'neutral'
    detected_category: 'subscription_cancellation'
    requires_more_info: False
    missing_fields: []
[ 0.33s] ---- rag_retrieve ----
    retrieved_chunks: [{'source': 'subscription_policy.md', 'text': '## Plan Changes\n- Customers may switch plans at any time from the billing settings page.\n- Plan changes take effect at the next billing cycle.\n\n## Escalation Guidance\nEscalate when:\n- the customer disputes a renewal charge,\n- the account shows a failed payment but the customer disputes the charge,\n- or the customer requests a special exception outside standard billing policy.', 'score': 0.47582459449768066}, {'source': 'subscription_policy.md', 'text': '# Subscription Policy\n\n## Billing and Renewal\n- Monthly and annual subscriptions renew automatically unless the customer cancels before the renewal date.\n- Cancellation requests are effective at the end of the current billing period

[ 0.76s] ---- draft_answer ----
    draft_reply: 'Draft Reply:\n\nDear Customer,\n\nYou can switch your subscription plan at any time from the billing settings page [source: subscription_policy.md]. To change your plan from annual to monthly billing for subscription SUB-5567, please navigate to the billing settings page and select the monthly plan option. Please note that plan changes take effect at the next billing cycle [source: subscription_policy.md].\n\nIf you have any further questions or need assistance with changing your plan, please let us know.\n\nBest regards,\n[Your Name]'
    groundedness_score: 0.47582459449768066
    fabricated_citations: []
[ 0.76s] ---- route ----
    route_decision: 'AUTO_RESOLVE'
    route_reason: 'policy_grounded_response'
[ 0.92s] ---- confidence_recheck ----
    llm_groundedness_score: 1.0
    unsupported_claims: []
    llm_groundedness_passed: True
    query_reformulation_hint: ''
[ 0.93s] ---- hitl_gate ----
    reviewer_action: 'APPROVED'
    r

## Final state

Everything accumulated across the run — the same shape
`outputs/results/TCK-1016.json` would hold after a CLI run.

In [5]:
print(f"route_decision: {final_state.get('route_decision')} ({final_state.get('route_reason')})")
print(f"groundedness_score (retrieval): {final_state.get('groundedness_score')}")
print(f"llm_groundedness_score (judge): {final_state.get('llm_groundedness_score')}")
print(f"retrieval_attempts: {final_state.get('retrieval_attempts')}")
print(f"reviewer_action: {final_state.get('reviewer_action')}")
print(f"\ndraft_reply:\n{final_state.get('draft_reply')}")

route_decision: AUTO_RESOLVE (policy_grounded_response)
groundedness_score (retrieval): 0.47582459449768066
llm_groundedness_score (judge): 1.0
retrieval_attempts: 1
reviewer_action: APPROVED

draft_reply:
Draft Reply:

Dear Customer,

You can switch your subscription plan at any time from the billing settings page [source: subscription_policy.md]. To change your plan from annual to monthly billing for subscription SUB-5567, please navigate to the billing settings page and select the monthly plan option. Please note that plan changes take effect at the next billing cycle [source: subscription_policy.md].

If you have any further questions or need assistance with changing your plan, please let us know.

Best regards,
[Your Name]


## Summary

Watching the stream rather than reading a persisted snapshot makes two
things visible that a static JSON file hides:

- **Timing** — where the pipeline actually spends time (the two
  LLM-backed nodes, `draft_answer` and `confidence_recheck`, dominate;
  the deterministic rule nodes are near-instant).
- **The conditional branch firing live** — `route` resolving to
  `AUTO_RESOLVE` is what sends this run into `confidence_recheck` at all;
  any other route would have skipped straight to `hitl_gate`, as
  `single_ticket_walkthrough.ipynb`'s `TCK-1002`/`TCK-1005`/`TCK-1009`
  examples show.